### FASE 3: Ingenieria de Caracteristicas (Creacion de KPIs)

In [7]:
import pandas as pd
import numpy as np

In [3]:
equip_rep_clean=pd.read_csv("/workspaces/ESTADISTICA_sem_II_2025/Proyecto_1/DATA/equip_rep_clean.csv")

In [11]:
def crear_features_avanzadas(df):
    """
    Crea features avanzadas de análisis de fútbol.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame con las estadísticas base del equipo
        
    Returns:
    --------
    pd.DataFrame con las nuevas features añadidas
    """
    
    return df.assign(
        #1. PRECISIÓN DE TIRO
        #Mide qué tan bien apuntan los tiros (% que van al arco)
        **{
            'precision_tiro': lambda x: np.where(
                x['totalShots'] > 0,
                x['shotsOnTarget'] / x['totalShots'] * 100,
                0
            )
        },
        
        #2. APROVECHAMIENTO DE POSESIÓN
        #Tiros generados por cada 1% de posesión
        ** {
            'aprovechamiento_posesion': lambda x: np.where(
                x['possessionPct'] > 0,
                x['totalShots'] / x['possessionPct'],
                0
            )
        },
        
        #Versión con tiros al arco (más estricta)
        ** {
            'aprovechamiento_posesion_peligroso': lambda x: np.where(
                x['possessionPct'] > 0,
                x['shotsOnTarget'] / x['possessionPct'],
                0
            )
        },
        # 3. PELIGROSIDAD EN CENTROS
        # Combina precisión de centros con capacidad de remate
        **{
            'peligrosidad_centros': lambda x: np.where(
                x['totalCrosses'] > 0,
                (x['accurateCrosses'] / x['totalCrosses']) * x['shotPct'] / 100,
                0
            )
        },
        
        # Ratio simple de centros precisos
        **{
            'precision_centros': lambda x: np.where(
                x['totalCrosses'] > 0,
                x['accurateCrosses'] / x['totalCrosses'] * 100,
                0
            )
        },
        
        # 4. ÍNDICE DE LIMPIEZA DEFENSIVA
        # % de recuperaciones sin faltas
        **{
            'limpieza_defensiva': lambda x: np.where(
                (x['interceptions'] + x['foulsCommitted']) > 0,
                x['interceptions'] / (x['interceptions'] + x['foulsCommitted']) * 100,
                0
            )
        },
        
        # Versión incluyendo tackles
        **{
            'limpieza_defensiva_completa': lambda x: np.where(
                (x['effectiveTackles'] + x['interceptions'] + x['foulsCommitted']) > 0,
                (x['effectiveTackles'] + x['interceptions']) / 
                (x['effectiveTackles'] + x['interceptions'] + x['foulsCommitted']) * 100,
                0
            )
        },
        
        # 5. EFECTIVIDAD DE DESPEJE
        # % de despejes efectivos
        **{
            'efectividad_despeje': lambda x: np.where(
                x['totalClearance'] > 0,
                x['effectiveClearance'] / x['totalClearance'] * 100,
                0
            )
        },
        # 6. ÍNDICE DE VERTICALIDAD
        # % de pases que son largos (juego directo vs. toque)
        **{
            'indice_verticalidad': lambda x: np.where(
                x['totalPasses'] > 0,
                x['totalLongBalls'] / x['totalPasses'] * 100,
                0
            )
        },
        
        # Verticalidad efectiva (solo balones largos precisos)
        **{
            'verticalidad_efectiva': lambda x: np.where(
                x['totalPasses'] > 0,
                x['accurateLongBalls'] / x['totalPasses'] * 100,
                0
            )
        },
        
        # Precisión en verticalidad
        **{
            'precision_verticalidad': lambda x: np.where(
                x['totalLongBalls'] > 0,
                x['accurateLongBalls'] / x['totalLongBalls'] * 100,
                0
            )
        },
        
        # 7. INTENSIDAD DE CIRCULACIÓN
        # Pases por cada 1% de posesión
        **{
            'intensidad_circulacion': lambda x: np.where(
                x['possessionPct'] > 0,
                x['totalPasses'] / x['possessionPct'],
                0
            )
        },
        
        # Solo pases precisos
        **{
            'intensidad_circulacion_precisa': lambda x: np.where(
                x['possessionPct'] > 0,
                x['accuratePasses'] / x['possessionPct'],
                0
            )
        }
        )

In [12]:
equip_rep_clean_kpi = crear_features_avanzadas (equip_rep_clean)

In [14]:
calculated_columns = [
    'slug',
    'precision_tiro',
    'aprovechamiento_posesion',
    'aprovechamiento_posesion_peligroso',
    'peligrosidad_centros',
    'precision_centros',
    'limpieza_defensiva',
    'limpieza_defensiva_completa',
    'efectividad_despeje',
    'indice_verticalidad',
    'verticalidad_efectiva',
    'precision_verticalidad',
    'intensidad_circulacion',
    'intensidad_circulacion_precisa'
]
equip_rep_clean_kpi[calculated_columns].groupby("slug").agg("mean")

,precision_tiro,aprovechamiento_posesion,aprovechamiento_posesion_peligroso,peligrosidad_centros,precision_centros,limpieza_defensiva,limpieza_defensiva_completa,efectividad_despeje,indice_verticalidad,verticalidad_efectiva,precision_verticalidad,intensidad_circulacion,intensidad_circulacion_precisa
slug,,,,,,,,,,,,,
eng.arsenal,39.851262,0.254288,0.099733,0.001023,24.938827,38.390011,59.566358,100.0,8.534320,3.845224,46.313695,8.529185,7.330412
eng.chelsea,38.208574,0.263577,0.103578,0.000956,23.969611,40.142756,59.800812,100.0,8.053281,3.872813,49.978795,9.357115,8.240882
eng.liverpool,37.804479,0.271908,0.101547,0.000848,21.545329,41.150743,60.924357,100.0,8.972039,4.159367,47.696029,9.197604,7.923767
eng.man_city,39.369714,0.263842,0.102424,0.000994,24.931187,41.867150,62.814214,100.0,6.052722,3.228861,55.152469,9.756807,8.752099
esp.atletico_madrid,41.306924,0.244796,0.102962,0.000898,22.461563,40.146891,61.421891,100.0,9.571053,5.051942,53.610276,9.923830,8.455840
esp.barcelona,39.411862,0.262502,0.103242,0.000867,22.000689,39.607412,62.214116,100.0,6.588130,3.679545,56.494395,9.445931,8.360345
esp.real_madrid,39.536040,0.301649,0.119645,0.000820,20.756735,46.320865,65.669250,100.0,8.169113,4.736392,59.870214,10.032205,8.959381
fra.psg,42.487609,0.272083,0.111654,0.001073,25.455369,44.202228,65.776640,100.0,6.365902,3.880242,63.446487,10.332159,9.366354
ger.bayern_munich,40.679268,0.292728,0.120547,0.000920,23.076201,43.654105,63.350523,100.0,7.068960,4.337735,63.826004,10.236684,9.151464
